# Preparación: IDs consistentes + edges dirigidas

In [ ]:
from pyspark.sql import functions as F, Window

# ---------- helpers ----------
def normalize_ids(df_clients, df_tranfs):
    # set de cuentas internas
    internal_accounts = df_clients.select(F.col("contrato_id").cast("string").alias("cta")).distinct()

    t = (df_tranfs
         .select(
             F.col("event_date"),
             F.col("mto_trf").cast("double").alias("mto_trf"),
             F.col("cta_ori").cast("string").alias("cta_ori"),
             F.col("rut_ori").cast("string").alias("rut_ori"),
             F.col("cta_dst").cast("string").alias("cta_dst"),
             F.col("rut_dst").cast("string").alias("rut_dst"),
             F.col("bco_dst").cast("string").alias("bco_dst"),
             F.col("tip_cta").cast("string").alias("tip_cta_dst"),
             F.col("trc_num").cast("string").alias("trc_num"),
             F.col("cod_est").cast("string").alias("cod_est"),
         )
    )

    # marca si destino es interno
    t = (t.join(internal_accounts.withColumn("dst_is_internal", F.lit(1)),
                t["cta_dst"] == internal_accounts["cta"], "left")
           .drop("cta")
           .withColumn("dst_is_internal", F.coalesce("dst_is_internal", F.lit(0)))
    )

    # account keys
    t = (t
         .withColumn("src_acct", F.concat(F.lit("INT:"), F.col("cta_ori")))
         .withColumn(
             "dst_acct",
             F.when(F.col("dst_is_internal") == 1, F.concat(F.lit("INT:"), F.col("cta_dst")))
              .otherwise(F.concat(F.lit("EXT:"), F.coalesce("bco_dst", F.lit("NA")), F.lit(":"), F.col("cta_dst")))
         )
    )

    # tx_id estable (si trc_num es único global, úsalo; si no, hashea)
    t = t.withColumn(
        "tx_id",
        F.sha2(F.concat_ws("||", "trc_num", "event_date", "src_acct", "dst_acct", F.col("mto_trf").cast("string")), 256)
    )

    edges = (t.select(
        "tx_id", F.col("src_acct").alias("src"), F.col("dst_acct").alias("dst"),
        "event_date", "mto_trf", "cod_est", "bco_dst", "dst_is_internal",
        "rut_ori", "rut_dst", "tip_cta_dst", "trc_num"
    ).cache())

    return internal_accounts, edges

internal_accounts, edges = normalize_ids(df_clients, df_tranfs)


# Riesgo sin labels: edge_flag + risk por cuenta (rápido)

## Edge flags (monto extremo + “burst” diario + estados raros)

In [ ]:
def build_edge_flags(edges, q_hi=0.999, burst_q=0.999, cod_est_rare_min_pct=0.01):
    """
    edge_flag = 1 si:
      - mto_trf >= quantil alto global
      - o la cuenta origina "burst" diario (tx/día en quantil alto)
      - o cod_est es "raro" (no es de los más frecuentes)
    """
    # 1) umbral monto global (aprox)
    q = edges.approxQuantile("mto_trf", [q_hi], 0.01)[0]

    e = edges.withColumn("is_hi_amt", F.when(F.col("mto_trf") >= F.lit(q), 1).otherwise(0))

    # 2) burst por día y src
    e = e.withColumn("day", F.to_date("event_date"))
    daily = (e.groupBy("src", "day").agg(F.count("*").alias("tx_day_cnt")))
    burst_thr = daily.approxQuantile("tx_day_cnt", [burst_q], 0.01)[0]

    burst_src_day = (daily
                     .withColumn("is_burst", F.when(F.col("tx_day_cnt") >= F.lit(burst_thr), 1).otherwise(0))
                     .select("src", "day", "is_burst"))

    e = (e.join(burst_src_day, ["src", "day"], "left")
           .na.fill({"is_burst": 0}))

    # 3) cod_est raro (si hay códigos; útil como proxy de falla/anomalía)
    #    definimos "raro" como códigos fuera del top que cubre (100 - cod_est_rare_min_pct*100)%
    cod = edges.groupBy("cod_est").count()
    total = edges.count()
    cod = cod.withColumn("pct", F.col("count") / F.lit(total))

    # Nos quedamos con "raros": pct < umbral
    rare = cod.filter(F.col("pct") < F.lit(cod_est_rare_min_pct)).select("cod_est").withColumn("is_rare_cod", F.lit(1))
    e = (e.join(rare, "cod_est", "left")
           .withColumn("is_rare_cod", F.coalesce("is_rare_cod", F.lit(0))))

    e = e.withColumn(
        "edge_flag",
        F.when((F.col("is_hi_amt") == 1) | (F.col("is_burst") == 1) | (F.col("is_rare_cod") == 1), 1).otherwise(0)
    )

    return e.drop("day").cache()

edges_f = build_edge_flags(edges)


## Risk por cuenta (ratio suavizado, robusto a rareza)

In [ ]:
def account_risk(edges_f, alpha=1.0, beta=1000.0):
    out_stats = (edges_f.groupBy("src")
                 .agg(F.count("*").alias("out_cnt"),
                      F.sum("edge_flag").alias("out_flag"),
                      F.approx_count_distinct("dst").alias("out_cp")))
    in_stats  = (edges_f.groupBy("dst")
                 .agg(F.count("*").alias("in_cnt"),
                      F.sum("edge_flag").alias("in_flag"),
                      F.approx_count_distinct("src").alias("in_cp")))

    a = (out_stats.withColumnRenamed("src", "account")
         .join(in_stats.withColumnRenamed("dst", "account"), "account", "full")
         .na.fill(0)
         .withColumn("tx_cnt", F.col("out_cnt") + F.col("in_cnt"))
         .withColumn("flag_cnt", F.col("out_flag") + F.col("in_flag"))
         .withColumn("cp_cnt", F.col("out_cp") + F.col("in_cp"))
         .withColumn("risk", (F.col("flag_cnt") + F.lit(alpha)) / (F.col("tx_cnt") + F.lit(beta)))
         .cache())
    return a

acct = account_risk(edges_f)


# Semillas pos/neg (internas y algunas externas)

In [ ]:
def pick_seeds(acct, internal_accounts, n_pos=500, n_neg=500, min_activity=20, neg_risk_max=0.0005):
    internal_keys = internal_accounts.select(F.concat(F.lit("INT:"), F.col("cta")).alias("account")).distinct()

    acct_f = acct.filter(F.col("tx_cnt") >= F.lit(min_activity)).cache()

    # positivas: top risk (internas)
    pos = (acct_f.join(internal_keys, "account", "inner")
           .orderBy(F.col("risk").desc(), F.col("flag_cnt").desc(), F.col("tx_cnt").desc())
           .limit(n_pos)
           .select(F.col("account").alias("seed_account"))
           .withColumn("seed_label", F.lit(1)))

    # negativas: internas “tranquilas”
    neg = (acct_f.join(internal_keys, "account", "inner")
           .filter((F.col("flag_cnt") == 0) & (F.col("risk") <= F.lit(neg_risk_max)))
           .orderBy(F.rand())
           .limit(n_neg)
           .select(F.col("account").alias("seed_account"))
           .withColumn("seed_label", F.lit(0)))

    # opcional: agregar algunas externas (para cumplir “no todas de df_clients”)
    # tomamos externas con actividad moderada (para conectividad) y riesgo bajo (neg)
    ext = (acct_f.filter(~F.col("account").startswith("INT:"))
           .orderBy(F.col("tx_cnt").desc())
           .limit(int(0.2 * n_neg))
           .select(F.col("account").alias("seed_account"))
           .withColumn("seed_label", F.lit(0)))

    return pos.unionByName(neg).unionByName(ext).distinct().cache()

seeds = pick_seeds(acct, internal_accounts, n_pos=800, n_neg=800)


# Núcleo 1: Snowball BFS (rápido por frontier joins)

In [ ]:
def snowball_bfs(edges_df, seeds_df, max_hops=3, max_neighbors_per_node=20,
                 bidirectional=True, budget_nodes=None):
    """
    BFS estilo snowball: frontier expansion.
    - budget_nodes: si no es None, recorta frontier para no pasarse demasiado.
    """
    e = edges_df.select("tx_id", "src", "dst", "mto_trf", "event_date", "cod_est",
                        "bco_dst", "dst_is_internal", "edge_flag").cache()

    visited = seeds_df.select(F.col("seed_account").alias("account")).distinct().cache()
    frontier = visited
    sampled_edges = None

    for hop in range(1, max_hops + 1):
        out_n = (frontier.join(e, frontier["account"] == e["src"], "inner")
                 .select("tx_id","src","dst","mto_trf","event_date","cod_est","bco_dst","dst_is_internal","edge_flag",
                         frontier["account"].alias("frontier_account"),
                         e["dst"].alias("neighbor"),
                         F.lit(hop).alias("hop")))

        if bidirectional:
            in_n = (frontier.join(e, frontier["account"] == e["dst"], "inner")
                    .select("tx_id","src","dst","mto_trf","event_date","cod_est","bco_dst","dst_is_internal","edge_flag",
                            frontier["account"].alias("frontier_account"),
                            e["src"].alias("neighbor"),
                            F.lit(hop).alias("hop")))
            neigh = out_n.unionByName(in_n)
        else:
            neigh = out_n

        # cap vecinos por nodo/hop
        w = Window.partitionBy("frontier_account").orderBy(F.rand())
        neigh = (neigh.withColumn("rn", F.row_number().over(w))
                      .filter(F.col("rn") <= F.lit(max_neighbors_per_node))
                      .drop("rn")
                      .cache())

        sampled_edges = neigh if sampled_edges is None else sampled_edges.unionByName(neigh)

        new_nodes = neigh.select(F.col("neighbor").alias("account")).distinct()
        frontier = new_nodes.join(visited, "account", "left_anti").cache()
        visited = visited.unionByName(frontier).distinct().cache()

        if budget_nodes is not None:
            # recorte suave: si ya te pasaste mucho, corta expansión
            if visited.count() >= budget_nodes:
                break

        if frontier.rdd.isEmpty():
            break

    sampled_edges = (sampled_edges
                     .drop("frontier_account","neighbor")
                     .dropDuplicates(["tx_id"])
                     .cache())
    sampled_nodes = visited.cache()
    return sampled_nodes, sampled_edges


# Núcleo 2: Snowball SIN BFS (expansión fija 1–2 hops por joins)

In [ ]:
#
def oneshot_khop(edges_df, seeds_df, hops=2, max_neighbors_per_node=20):
    """
    Expansión fija (1 o 2 hops) por joins, sin BFS iterativo.
    """
    e = edges_df.select("tx_id","src","dst","mto_trf","event_date","cod_est","bco_dst","dst_is_internal","edge_flag").cache()
    seed_nodes = seeds_df.select(F.col("seed_account").alias("account")).distinct()

    # hop 1: incidentes (bidireccional)
    hop1 = (seed_nodes.join(e, seed_nodes["account"] == e["src"], "inner")
            .select(e["tx_id"], e["src"], e["dst"], seed_nodes["account"].alias("seed"))
            .unionByName(
                seed_nodes.join(e, seed_nodes["account"] == e["dst"], "inner")
                .select(e["tx_id"], e["src"], e["dst"], seed_nodes["account"].alias("seed"))
            ))

    w = Window.partitionBy("seed").orderBy(F.rand())
    hop1 = (hop1.withColumn("rn", F.row_number().over(w))
                .filter(F.col("rn") <= F.lit(max_neighbors_per_node))
                .drop("rn")
                .cache())

    nodes1 = (hop1.select(F.col("src").alias("account"))
                  .unionByName(hop1.select(F.col("dst").alias("account")))
                  .distinct()
                  .cache())

    if hops == 1:
        edges1 = hop1.select("tx_id","src","dst").distinct()
        return nodes1, edges_df.join(edges1, "tx_id", "inner")

    # hop 2: vecinos de nodes1 (otra vez bidireccional)
    hop2 = (nodes1.join(e, nodes1["account"] == e["src"], "inner")
            .select("tx_id","src","dst")
            .unionByName(
                nodes1.join(e, nodes1["account"] == e["dst"], "inner")
                .select("tx_id","src","dst")
            )
            .distinct()
            .cache())

    nodes2 = (nodes1.unionByName(hop2.select(F.col("src").alias("account")))
                    .unionByName(hop2.select(F.col("dst").alias("account")))
                    .distinct()
                    .cache())

    edges2 = edges_df.join(hop2, "tx_id", "inner").cache()
    return nodes2, edges2


# Heurística A: Semillas “conectables” por overlap de vecinos (SIN y CON BFS)

## A.1 Construir vecinos top-K por cuenta (rápido)

In [ ]:
def topk_neighbors(edges_df, k=30):
    # vecinos por src (frecuencia) + por dst (para simetría)
    out = (edges_df.groupBy("src", "dst").agg(F.count("*").alias("w"))
           .withColumnRenamed("src", "account")
           .withColumnRenamed("dst", "nbr"))
    inn = (edges_df.groupBy("dst", "src").agg(F.count("*").alias("w"))
           .withColumnRenamed("dst", "account")
           .withColumnRenamed("src", "nbr"))
    adj = out.unionByName(inn)

    w = Window.partitionBy("account").orderBy(F.col("w").desc())
    return (adj.withColumn("rn", F.row_number().over(w))
               .filter(F.col("rn") <= F.lit(k))
               .select("account","nbr")
               .cache())


## A.2 Seleccionar semillas maximizando overlap (aprox)

In [ ]:
# La idea: tomar seeds candidatas y quedarte con las que caen en el mismo “cluster” de overlap (menos islas).
def seeds_by_overlap(edges_df, seeds_df, k=30, overlap_min=2, n_final=1500):
    adj = topk_neighbors(edges_df, k=k)

    cand = seeds_df.select(F.col("seed_account").alias("account")).distinct()
    cand_adj = cand.join(adj, "account", "inner").cache()

    # overlap: semillas que comparten vecino (self-join por nbr)
    pairs = (cand_adj.alias("a").join(cand_adj.alias("b"), "nbr")
             .where(F.col("a.account") < F.col("b.account"))
             .groupBy(F.col("a.account").alias("s1"), F.col("b.account").alias("s2"))
             .agg(F.count("*").alias("shared_nbrs"))
             .filter(F.col("shared_nbrs") >= F.lit(overlap_min))
            )

    # score de conectabilidad: cuántos overlaps tiene cada seed
    score = (pairs.select(F.col("s1").alias("account"))
                  .unionByName(pairs.select(F.col("s2").alias("account")))
                  .groupBy("account").count()
                  .withColumnRenamed("count", "overlap_deg"))

    top = (cand.join(score, "account", "left").na.fill({"overlap_deg": 0})
           .orderBy(F.col("overlap_deg").desc())
           .limit(n_final))

    return top.select(F.col("account").alias("seed_account")).withColumn("seed_label", F.lit(9)).cache()


## A sin BFS (oneshot 2-hop)

In [ ]:
seeds_A = seeds_by_overlap(edges_f, seeds, k=30, overlap_min=2, n_final=2000)
nodes_A_nobfs, edges_A_nobfs = oneshot_khop(edges_f, seeds_A, hops=2, max_neighbors_per_node=15)


## A con BFS (snowball)

In [ ]:
nodes_A_bfs, edges_A_bfs = snowball_bfs(edges_f, seeds_A, max_hops=3, max_neighbors_per_node=7,
                                       bidirectional=True, budget_nodes=30000)


# Heurística B: Giant Component First (SIN y CON BFS)

In [ ]:
# Elegir “hub” (alto grado/actividad)
def pick_hub(acct, internal_accounts):
    internal_keys = internal_accounts.select(F.concat(F.lit("INT:"), F.col("cta")).alias("account")).distinct()
    hub = (acct.join(internal_keys, "account", "inner")
           .orderBy(F.col("tx_cnt").desc(), F.col("cp_cnt").desc())
           .limit(1)
           .select(F.col("account").alias("seed_account"))
           .withColumn("seed_label", F.lit(7)))
    return hub.cache()

hub = pick_hub(acct, internal_accounts)


## B sin BFS: construir subgrafo “hub + vecinos” y quedarte con el componente más grande (si tienes GraphFrames)

In [ ]:
nodes_B_nobfs, edges_B_nobfs = oneshot_khop(edges_f, hub, hops=2, max_neighbors_per_node=50)
# opcional: connectedComponents sobre nodes_B_nobfs/edges_B_nobfs (ver benchmark más abajo)


## B con BFS: construir componente C desde hub y luego muestrear dentro

In [ ]:
C_nodes, C_edges = snowball_bfs(edges_f, hub, max_hops=2, max_neighbors_per_node=60, bidirectional=True)

# re-muestrea seeds dentro de C para asegurar conectividad
seeds_in_C = (seeds.select(F.col("seed_account").alias("account"))
              .join(C_nodes, "account", "inner")
              .select(F.col("account").alias("seed_account"))
              .withColumn("seed_label", F.lit(8))
              .limit(2000)
              .cache())

nodes_B_bfs, edges_B_bfs = snowball_bfs(edges_f, seeds_in_C, max_hops=3, max_neighbors_per_node=10,
                                       bidirectional=True, budget_nodes=30000)


# Heurística C: Bridging para anexar seeds fuera del componente principal

## C con BFS (tipo “bfs_until_target”, anexión por lotes)

In [ ]:
def undirect_edges(edges_df):
    u = edges_df.select("tx_id","src","dst","mto_trf","event_date","cod_est","bco_dst","dst_is_internal","edge_flag")
    return u.unionByName(u.select("tx_id", F.col("dst").alias("src"), F.col("src").alias("dst"),
                                  "mto_trf","event_date","cod_est","bco_dst","dst_is_internal","edge_flag"))

def bfs_until_target(undir_edges, start_nodes_df, target_nodes_df, max_hops=3, max_neighbors_per_node=30):
    visited = start_nodes_df.select("account").distinct().cache()
    frontier = visited
    sampled_edges = None
    hit = False

    for hop in range(1, max_hops + 1):
        neigh = (frontier.join(undir_edges, frontier["account"] == undir_edges["src"], "inner")
                 .select(undir_edges["tx_id"], undir_edges["src"], undir_edges["dst"],
                         frontier["account"].alias("frontier"),
                         F.lit(hop).alias("hop"))
                )
        w = Window.partitionBy("frontier").orderBy(F.rand())
        neigh = (neigh.withColumn("rn", F.row_number().over(w))
                      .filter(F.col("rn") <= F.lit(max_neighbors_per_node))
                      .drop("rn")
                      .cache())

        sampled_edges = neigh if sampled_edges is None else sampled_edges.unionByName(neigh)

        new_nodes = neigh.select(F.col("dst").alias("account")).distinct()
        frontier = new_nodes.join(visited, "account", "left_anti").cache()
        visited = visited.unionByName(frontier).distinct().cache()

        if not frontier.join(target_nodes_df, "account", "inner").rdd.isEmpty():
            hit = True
            break
        if frontier.rdd.isEmpty():
            break

    if sampled_edges is None:
        sampled_edges = undir_edges.limit(0).select("tx_id","src","dst").withColumn("hop", F.lit(0))

    return visited, sampled_edges.select("tx_id","src","dst","hop").dropDuplicates(["tx_id"]), hit

def bridging_C_with_bfs(edges_df, seeds_df, hub_df, component_hops=2, bridge_hops=3,
                        max_neighbors_per_node=30, batch_size=200):
    u = undirect_edges(edges_df).cache()

    # componente principal C desde hub
    C_nodes, C_edges = snowball_bfs(u, hub_df, max_hops=component_hops, max_neighbors_per_node=80,
                                   bidirectional=False)  # u ya es no-dirigido
    C_nodes = C_nodes.cache()
    all_edges = C_edges.select("tx_id","src","dst").cache()

    seeds_nodes = seeds_df.select(F.col("seed_account").alias("account")).distinct()
    remaining = seeds_nodes.join(C_nodes, "account", "left_anti")

    # procesar por lotes (sin collect masivo)
    idx = remaining.withColumn("bucket", (F.rand()*1000000).cast("int"))
    buckets = [r["bucket"] for r in idx.select("bucket").distinct().limit(100).collect()]  # acota loops

    for b in buckets:
        batch = idx.filter(F.col("bucket") == b).limit(batch_size).select("account").cache()
        if batch.rdd.isEmpty():
            continue

        b_nodes, b_edges, hit = bfs_until_target(u, batch, C_nodes, max_hops=bridge_hops,
                                                 max_neighbors_per_node=max_neighbors_per_node)
        if hit:
            C_nodes = C_nodes.unionByName(b_nodes).distinct().cache()
            all_edges = all_edges.unionByName(b_edges.select("tx_id","src","dst")).dropDuplicates(["tx_id"]).cache()

    # devolver subgrafo conectado grande (C) y sus edges
    final_nodes = C_nodes.cache()
    final_edges = edges_df.join(all_edges, "tx_id", "inner").cache()
    return final_nodes, final_edges

nodes_C_bfs, edges_C_bfs = bridging_C_with_bfs(edges_f, seeds, hub, component_hops=2, bridge_hops=3,
                                              max_neighbors_per_node=20, batch_size=200)


## C sin BFS (puentes 1-hop y 2-hop por joins; sin búsqueda iterativa)

La idea: si tu muestra tiene componentes, intenta unirlas trayendo edges del grafo completo que conecten:

- directo (1-hop) a la componente principal
- o vía un intermediario (2-hop) con dos joins

In [ ]:
def bridge_twohop_no_bfs(edges_df, main_nodes, other_nodes):
    """
    Intenta anexar otros_nodes al main_nodes usando:
      - 1-hop: edge src in other & dst in main  OR src in main & dst in other
      - 2-hop: other -> x -> main (x se añade)
    """
    e = edges_df.select("tx_id","src","dst").cache()
    main = main_nodes.select("account").distinct().cache()
    other = other_nodes.select("account").distinct().cache()

    # 1-hop bridges
    one = (e.join(other, e["src"] == other["account"], "inner")
             .join(main, e["dst"] == main["account"], "inner")
             .select("tx_id","src","dst")
           ).unionByName(
           e.join(main, e["src"] == main["account"], "inner")
            .join(other, e["dst"] == other["account"], "inner")
            .select("tx_id","src","dst")
           ).distinct().cache()

    # si no hay 1-hop, intentar 2-hop: other -> x y x -> main
    # paso A: fronteras desde other
    stepA = (e.join(other, e["src"] == other["account"], "inner")
               .select(e["dst"].alias("x"))
               .distinct().cache())

    # paso B: x -> main
    two = (e.join(stepA, e["src"] == stepA["x"], "inner")
             .join(main, e["dst"] == main["account"], "inner")
             .select("tx_id","src","dst")
          ).distinct().cache()

    bridges = one.unionByName(two).distinct().cache()

    # nodos añadidos por los puentes
    add_nodes = (bridges.select(F.col("src").alias("account"))
                        .unionByName(bridges.select(F.col("dst").alias("account")))
                        .distinct())

    return add_nodes, bridges

# ejemplo de uso: partir con un "main" del hub, y anexar seeds fuera con 2-hop
main_nodes0, main_edges0 = oneshot_khop(edges_f, hub, hops=2, max_neighbors_per_node=80)
remaining_seeds = seeds.select(F.col("seed_account").alias("account")).join(main_nodes0, "account", "left_anti")

add_nodes, bridges = bridge_twohop_no_bfs(edges_f, main_nodes0, remaining_seeds)
nodes_C_nobfs = main_nodes0.unionByName(add_nodes).distinct().cache()
edges_C_nobfs = edges_f.join(bridges, "tx_id", "inner").cache()


# Benchmark básico (tamaño + conectividad si GraphFrames existe)

In [ ]:
import time

def connectivity_metrics(sampled_nodes, sampled_edges):
    """
    Si tienes GraphFrames:
      - num_components (weak connectivity)
      - largest_component_size
      - pct_in_largest
    """
    try:
        from graphframes import GraphFrame
    except Exception:
        return {"components": None, "largest": None, "pct_largest": None}

    v = sampled_nodes.select(F.col("account").alias("id")).distinct()
    # weak connectivity => tratamos edges como no-dirigidas duplicando
    e = sampled_edges.select("src","dst").distinct()
    g = GraphFrame(v, e)

    cc = g.connectedComponents()  # no dirigido
    comp_sizes = cc.groupBy("component").count()

    total = cc.count()
    largest = comp_sizes.orderBy(F.col("count").desc()).limit(1).collect()[0]["count"]
    components = comp_sizes.count()
    return {"components": components, "largest": largest, "pct_largest": float(largest) / float(total) if total else None}

def bench_one(name, nodes_df, edges_df):
    t0 = time.perf_counter()
    n_nodes = nodes_df.count()
    n_edges = edges_df.count()
    met = connectivity_metrics(nodes_df, edges_df)
    t1 = time.perf_counter()
    return {
        "name": name,
        "nodes": n_nodes,
        "edges": n_edges,
        "components": met["components"],
        "largest_comp": met["largest"],
        "pct_in_largest": met["pct_largest"],
        "secs": (t1 - t0),
    }

results = []

results.append(bench_one("A_noBFS", nodes_A_nobfs, edges_A_nobfs))
results.append(bench_one("A_BFS",   nodes_A_bfs,   edges_A_bfs))
results.append(bench_one("B_noBFS", nodes_B_nobfs, edges_B_nobfs))
results.append(bench_one("B_BFS",   nodes_B_bfs,   edges_B_bfs))
results.append(bench_one("C_noBFS", nodes_C_nobfs, edges_C_nobfs))
results.append(bench_one("C_BFS",   nodes_C_bfs,   edges_C_bfs))

spark.createDataFrame(results).orderBy(F.col("pct_in_largest").desc(), F.col("nodes").desc()).show(truncate=False)


# Escribir CSV

In [ ]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

def write_csv_paged(
    df,
    base_dir: str,
    name: str,
    page_size: int = 10_000,
    mode: str = "overwrite",
    header: bool = True,
    sep: str = ",",
    compression: str | None = None,   # e.g. "gzip"
    pad: int = 6,
    drop_index_cols: bool = True,
):
    """
    Guarda df en base_dir/name/ como N CSVs paginados:
      name_000001.csv, name_000002.csv, ...

    - page_size: filas por archivo (exacto salvo última página)
    - compression: None o "gzip"
    - mode: "overwrite" (recomendado) o "append" (ojo con colisiones)
    """

    spark = df.sparkSession
    jvm = spark._jvm
    hconf = spark._jsc.hadoopConfiguration()
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
    Path = jvm.org.apache.hadoop.fs.Path

    def _p(p):  # Path helper
        return Path(p)

    out_dir = f"{base_dir.rstrip('/')}/{name}"
    tmp_root = f"{out_dir}/_tmp_pages"

    # Manejo overwrite / append
    if mode.lower() == "overwrite":
        if fs.exists(_p(out_dir)):
            fs.delete(_p(out_dir), True)
        fs.mkdirs(_p(out_dir))
    else:
        if not fs.exists(_p(out_dir)):
            fs.mkdirs(_p(out_dir))

    # Limpia tmp
    if fs.exists(_p(tmp_root)):
        fs.delete(_p(tmp_root), True)
    fs.mkdirs(_p(tmp_root))

    t0 = time.perf_counter()

    # 1) Index contiguo (sin sort global) + page
    # zipWithIndex asigna índices por orden de particiones; suficiente para chunking
    cols = df.columns
    schema = df.schema.add("__idx", LongType(), nullable=False)

    rdd_idx = df.rdd.zipWithIndex().map(lambda ri: tuple(ri[0]) + (ri[1],))
    df_idx = spark.createDataFrame(rdd_idx, schema=schema)

    df_idx = df_idx.withColumn("__page", (F.col("__idx") / F.lit(page_size)).cast("long") + F.lit(1))

    # 2) Conteo y nº de páginas
    total_rows = df_idx.count()
    num_pages = (total_rows + page_size - 1) // page_size

    # 3) Escritura por página + rename del part file
    bytes_total = 0
    files_written = 0

    for pnum in range(1, num_pages + 1):
        tmp_dir = f"{tmp_root}/page={pnum}"
        final_ext = ".csv.gz" if compression else ".csv"
        final_file = f"{out_dir}/{name}_{str(pnum).zfill(pad)}{final_ext}"

        # filtra página
        page_df = df_idx.filter(F.col("__page") == F.lit(pnum))
        if drop_index_cols:
            page_df = page_df.drop("__idx", "__page")

        # escribe a tmp (un solo part)
        writer = (page_df.coalesce(1)
                  .write.mode("overwrite")
                  .option("header", str(header).lower())
                  .option("sep", sep))
        if compression:
            writer = writer.option("compression", compression)

        writer.csv(tmp_dir)

        # encuentra el part-*.csv(.gz) y renómbralo al nombre final
        statuses = fs.listStatus(_p(tmp_dir))
        part_path = None
        for st in statuses:
            pth = st.getPath().toString()
            # Spark suele producir part-*.csv o part-*.csv.gz según compresión
            if "/part-" in pth and (pth.endswith(".csv") or pth.endswith(".csv.gz")):
                part_path = pth
                part_len = st.getLen()
                break

        if part_path is None:
            raise RuntimeError(f"No encontré part file en {tmp_dir}. ¿Se escribió algo?")

        # Si existe final_file y mode != overwrite, evita colisión
        if fs.exists(_p(final_file)):
            fs.delete(_p(final_file), True)

        ok = fs.rename(_p(part_path), _p(final_file))
        if not ok:
            raise RuntimeError(f"No se pudo renombrar {part_path} -> {final_file}")

        # borra tmp_dir (incluye _SUCCESS etc.)
        fs.delete(_p(tmp_dir), True)

        bytes_total += part_len
        files_written += 1

    # borra tmp_root
    fs.delete(_p(tmp_root), True)

    t1 = time.perf_counter()

    mb_total = bytes_total / (1024 * 1024)
    secs = t1 - t0
    rps = total_rows / secs if secs > 0 else None

    print("=== write_csv_paged summary ===")
    print(f"Output dir        : {out_dir}")
    print(f"Base name         : {name}")
    print(f"Total rows        : {total_rows:,}")
    print(f"Page size         : {page_size:,}")
    print(f"Files written     : {files_written:,}")
    print(f"Total size (MB)   : {mb_total:,.2f}")
    print(f"Elapsed (sec)     : {secs:,.2f}")
    if rps is not None:
        print(f"Rows/sec approx   : {rps:,.0f}")
    print("===============================")

    return {
        "out_dir": out_dir,
        "name": name,
        "total_rows": total_rows,
        "page_size": page_size,
        "files_written": files_written,
        "bytes_total": bytes_total,
        "elapsed_sec": secs,
    }


In [ ]:
# Ejemplos de uso
# Para edges muestreadas
stats_edges = write_csv_paged(
    df=edges_A_bfs,              # o edges_B_bfs, edges_C_nobfs, etc.
    base_dir="/mnt/exports",
    name="edges_A_bfs",
    page_size=10_000,
    compression="gzip"           # opcional
)

# Para nodes muestreados (quizá páginas más grandes)
stats_nodes = write_csv_paged(
    df=nodes_A_bfs,
    base_dir="/mnt/exports",
    name="nodes_A_bfs",
    page_size=50_000
)


# Versión DFS

In [ ]:
import time
from pyspark.sql import functions as F, Window
from pyspark.storagelevel import StorageLevel

# ============================================================
# 1) DFS snowball (depth-first biased) en Spark
# ============================================================
def snowball_dfs(
    edges_df,
    seeds_df,
    max_depth=3,
    max_neighbors_per_node=20,
    batch_expand=200,          # cuántos nodos "pop" del stack por iteración
    bidirectional=True,
    budget_nodes=None,
    seed_col="seed_account"
):
    """
    DFS aproximado (depth-first biased):
      - Mantiene un stack de nodos por expandir (DataFrame)
      - Pop: toma batch_expand nodos con mayor (depth, order) (LIFO aprox)
      - Expande vecinos y los pushea con depth+1 y order creciente

    Retorna: (sampled_nodes_df[account], sampled_edges_df[tx_id,...])
    """
    spark = edges_df.sparkSession

    e = edges_df.select(
        "tx_id", "src", "dst", "mto_trf", "event_date", "cod_est",
        "bco_dst", "dst_is_internal", "edge_flag"
    ).persist(StorageLevel.MEMORY_AND_DISK)

    # visited
    visited = seeds_df.select(F.col(seed_col).alias("account")).distinct() \
        .withColumn("depth", F.lit(0)).persist(StorageLevel.MEMORY_AND_DISK)

    # stack: nodos por expandir (únicos por account)
    stack = visited.select("account", "depth") \
        .withColumn("order", F.monotonically_increasing_id()) \
        .persist(StorageLevel.MEMORY_AND_DISK)

    sampled_edges = None
    iter_no = 0

    while True:
        # Pop un batch del stack (más profundo / más reciente primero)
        batch = (stack.orderBy(F.col("depth").desc(), F.col("order").desc())
                      .limit(batch_expand)
                      .persist(StorageLevel.MEMORY_AND_DISK))

        if batch.rdd.isEmpty():
            break

        # Remover batch del stack
        stack = (stack.join(batch.select("account").distinct(), "account", "left_anti")
                      .persist(StorageLevel.MEMORY_AND_DISK))

        # Expandir vecinos desde batch
        # Salientes
        out_n = (batch.join(e, batch["account"] == e["src"], "inner")
                      .select(
                          e["tx_id"], e["src"], e["dst"], e["mto_trf"], e["event_date"], e["cod_est"],
                          e["bco_dst"], e["dst_is_internal"], e["edge_flag"],
                          batch["account"].alias("frontier_account"),
                          e["dst"].alias("neighbor"),
                          (batch["depth"] + F.lit(1)).alias("ndepth")
                      ))

        if bidirectional:
            # Entrantes
            in_n = (batch.join(e, batch["account"] == e["dst"], "inner")
                         .select(
                             e["tx_id"], e["src"], e["dst"], e["mto_trf"], e["event_date"], e["cod_est"],
                             e["bco_dst"], e["dst_is_internal"], e["edge_flag"],
                             batch["account"].alias("frontier_account"),
                             e["src"].alias("neighbor"),
                             (batch["depth"] + F.lit(1)).alias("ndepth")
                         ))
            neigh = out_n.unionByName(in_n)
        else:
            neigh = out_n

        # No pasar max_depth
        neigh = neigh.filter(F.col("ndepth") <= F.lit(max_depth))

        # Cap de vecinos por nodo expandido (frontier_account)
        w = Window.partitionBy("frontier_account").orderBy(F.rand())
        neigh = (neigh.withColumn("rn", F.row_number().over(w))
                      .filter(F.col("rn") <= F.lit(max_neighbors_per_node))
                      .drop("rn")
                      .persist(StorageLevel.MEMORY_AND_DISK))

        # Acumular edges
        sampled_edges = neigh if sampled_edges is None else sampled_edges.unionByName(neigh)

        # Descubrir nuevos nodos
        new_nodes = (neigh.select(F.col("neighbor").alias("account"), F.col("ndepth").alias("depth"))
                          .groupBy("account").agg(F.max("depth").alias("depth")))

        # Solo los no visitados
        new_nodes = (new_nodes.join(visited.select("account").distinct(), "account", "left_anti")
                              .persist(StorageLevel.MEMORY_AND_DISK))

        # Push al stack (LIFO aprox: order creciente)
        if not new_nodes.rdd.isEmpty():
            pushed = (new_nodes.withColumn("order", F.monotonically_increasing_id())
                              .persist(StorageLevel.MEMORY_AND_DISK))
            stack = (stack.unionByName(pushed.select("account", "depth", "order"))
                          .dropDuplicates(["account"])
                          .persist(StorageLevel.MEMORY_AND_DISK))

            visited = (visited.unionByName(new_nodes.select("account", "depth"))
                            .groupBy("account").agg(F.max("depth").alias("depth"))
                            .persist(StorageLevel.MEMORY_AND_DISK))

        iter_no += 1

        # budget (stop suave)
        if budget_nodes is not None:
            if visited.count() >= budget_nodes:
                break

    # Formato final
    sampled_nodes = visited.select("account").distinct().persist(StorageLevel.MEMORY_AND_DISK)
    if sampled_edges is None:
        sampled_edges = e.limit(0)

    sampled_edges = (sampled_edges
                     .drop("frontier_account", "neighbor", "ndepth")
                     .dropDuplicates(["tx_id"])
                     .persist(StorageLevel.MEMORY_AND_DISK))

    return sampled_nodes, sampled_edges


# ============================================================
# 2) DFS "hasta tocar un target" (para bridging de C)
# ============================================================
def dfs_until_target(
    undir_edges_df,            # edges ya "no dirigidos" (src->dst duplicado)
    start_nodes_df,            # DF(account)
    target_nodes_df,           # DF(account)
    max_depth=3,
    max_neighbors_per_node=20,
    batch_expand=200
):
    """
    DFS biased: expande desde start hasta tocar cualquier account en target.
    Retorna: (visited_nodes(account), visited_edges(tx_id,src,dst), hit_bool)
    """
    e = undir_edges_df.select("tx_id","src","dst").persist(StorageLevel.MEMORY_AND_DISK)
    target = target_nodes_df.select("account").distinct().persist(StorageLevel.MEMORY_AND_DISK)

    visited = start_nodes_df.select("account").distinct().withColumn("depth", F.lit(0)) \
        .persist(StorageLevel.MEMORY_AND_DISK)
    stack = visited.select("account","depth") \
        .withColumn("order", F.monotonically_increasing_id()) \
        .persist(StorageLevel.MEMORY_AND_DISK)

    sampled_edges = None
    hit = False

    while True:
        batch = (stack.orderBy(F.col("depth").desc(), F.col("order").desc())
                      .limit(batch_expand)
                      .persist(StorageLevel.MEMORY_AND_DISK))
        if batch.rdd.isEmpty():
            break

        stack = (stack.join(batch.select("account").distinct(), "account", "left_anti")
                      .persist(StorageLevel.MEMORY_AND_DISK))

        neigh = (batch.join(e, batch["account"] == e["src"], "inner")
                      .select(e["tx_id"], e["src"], e["dst"],
                              batch["account"].alias("frontier"),
                              (batch["depth"] + F.lit(1)).alias("ndepth")))

        neigh = neigh.filter(F.col("ndepth") <= F.lit(max_depth))

        w = Window.partitionBy("frontier").orderBy(F.rand())
        neigh = (neigh.withColumn("rn", F.row_number().over(w))
                      .filter(F.col("rn") <= F.lit(max_neighbors_per_node))
                      .drop("rn")
                      .persist(StorageLevel.MEMORY_AND_DISK))

        sampled_edges = neigh if sampled_edges is None else sampled_edges.unionByName(neigh)

        new_nodes = (neigh.select(F.col("dst").alias("account"), F.col("ndepth").alias("depth"))
                          .groupBy("account").agg(F.max("depth").alias("depth")))
        new_nodes = (new_nodes.join(visited.select("account"), "account", "left_anti")
                              .persist(StorageLevel.MEMORY_AND_DISK))

        # hit check (barato: join + limit 1)
        if not hit:
            if not new_nodes.join(target, "account", "inner").limit(1).rdd.isEmpty():
                hit = True
                # aún anexamos para incluir el "puente"
                visited = (visited.unionByName(new_nodes.select("account","depth"))
                                .groupBy("account").agg(F.max("depth").alias("depth"))
                                .persist(StorageLevel.MEMORY_AND_DISK))
                break

        if new_nodes.rdd.isEmpty():
            continue

        pushed = new_nodes.withColumn("order", F.monotonically_increasing_id())
        stack = (stack.unionByName(pushed.select("account","depth","order"))
                      .dropDuplicates(["account"])
                      .persist(StorageLevel.MEMORY_AND_DISK))

        visited = (visited.unionByName(new_nodes.select("account","depth"))
                        .groupBy("account").agg(F.max("depth").alias("depth"))
                        .persist(StorageLevel.MEMORY_AND_DISK))

    visited_nodes = visited.select("account").distinct()
    if sampled_edges is None:
        sampled_edges = e.limit(0).withColumn("frontier", F.lit(None)).withColumn("ndepth", F.lit(0))

    visited_edges = sampled_edges.select("tx_id","src","dst").dropDuplicates(["tx_id"])
    return visited_nodes, visited_edges, hit


# ============================================================
# 3) Heurísticas A/B/C: wrappers BFS vs DFS
#    (Asumo que ya tienes estas funciones del trabajo previo:
#      - seeds_by_overlap(...)
#      - pick_hub(...)
#      - undirect_edges(...)
#      - snowball_bfs(...)
#    y que tienes edges_f, seeds, acct, internal_accounts.
# ============================================================

def run_A_BFS(edges_f, seeds, budget_nodes=30000):
    seeds_A = seeds_by_overlap(edges_f, seeds, k=30, overlap_min=2, n_final=2000)
    return snowball_bfs(edges_f, seeds_A, max_hops=3, max_neighbors_per_node=10,
                        bidirectional=True, budget_nodes=budget_nodes)

def run_A_DFS(edges_f, seeds, budget_nodes=30000):
    seeds_A = seeds_by_overlap(edges_f, seeds, k=30, overlap_min=2, n_final=2000)
    return snowball_dfs(edges_f, seeds_A, max_depth=3, max_neighbors_per_node=10,
                        batch_expand=300, bidirectional=True, budget_nodes=budget_nodes)

def run_B_BFS(edges_f, acct, internal_accounts, budget_nodes=30000):
    hub = pick_hub(acct, internal_accounts)  # DF(seed_account, seed_label)
    return snowball_bfs(edges_f, hub, max_hops=3, max_neighbors_per_node=25,
                        bidirectional=True, budget_nodes=budget_nodes)

def run_B_DFS(edges_f, acct, internal_accounts, budget_nodes=30000):
    hub = pick_hub(acct, internal_accounts)
    return snowball_dfs(edges_f, hub, max_depth=3, max_neighbors_per_node=25,
                        batch_expand=300, bidirectional=True, budget_nodes=budget_nodes)

def run_C_BFS(edges_f, seeds, acct, internal_accounts, budget_nodes=30000):
    hub = pick_hub(acct, internal_accounts)
    # tu función previa bridging_C_with_bfs(...)
    nodes, eds = bridging_C_with_bfs(
        edges_df=edges_f, seeds_df=seeds, hub_df=hub,
        component_hops=2, bridge_hops=3, max_neighbors_per_node=15, batch_size=200
    )
    # recorte por budget si se pasó
    if budget_nodes is not None and nodes.count() > budget_nodes:
        # recorte simple: quedarte con los primeros budget_nodes (aprox)
        keep = nodes.limit(budget_nodes)
        eds = eds.join(keep.select(F.col("account").alias("src")), "src", "inner") \
                 .join(keep.select(F.col("account").alias("dst")), "dst", "inner")
        nodes = keep
    return nodes, eds

def bridging_C_with_dfs(edges_df, seeds_df, hub_df,
                        component_depth=2, bridge_depth=3,
                        max_neighbors_per_node=15, batch_size=200):
    u = undirect_edges(edges_df).select("tx_id","src","dst").persist(StorageLevel.MEMORY_AND_DISK)

    # componente principal con DFS desde hub
    C_nodes, C_edges = snowball_dfs(u, hub_df, max_depth=component_depth,
                                    max_neighbors_per_node=80, batch_expand=400,
                                    bidirectional=False, budget_nodes=None)
    C_nodes = C_nodes.persist(StorageLevel.MEMORY_AND_DISK)
    all_edges = C_edges.select("tx_id","src","dst").persist(StorageLevel.MEMORY_AND_DISK)

    seeds_nodes = seeds_df.select(F.col("seed_account").alias("account")).distinct()
    remaining = seeds_nodes.join(C_nodes, "account", "left_anti") \
                          .persist(StorageLevel.MEMORY_AND_DISK)

    # Batches: bucket aleatorio
    idx = remaining.withColumn("bucket", (F.rand() * 1000000).cast("int"))
    buckets = [r["bucket"] for r in idx.select("bucket").distinct().limit(200).collect()]

    for b in buckets:
        batch = idx.filter(F.col("bucket") == b).limit(batch_size).select("account")
        if batch.rdd.isEmpty():
            continue

        b_nodes, b_edges, hit = dfs_until_target(
            u, batch, C_nodes, max_depth=bridge_depth,
            max_neighbors_per_node=max_neighbors_per_node, batch_expand=400
        )
        if hit:
            C_nodes = C_nodes.unionByName(b_nodes).distinct().persist(StorageLevel.MEMORY_AND_DISK)
            all_edges = all_edges.unionByName(b_edges).dropDuplicates(["tx_id"]).persist(StorageLevel.MEMORY_AND_DISK)

    final_nodes = C_nodes.persist(StorageLevel.MEMORY_AND_DISK)
    final_edges = edges_df.join(all_edges, "tx_id", "inner").persist(StorageLevel.MEMORY_AND_DISK)
    return final_nodes, final_edges

def run_C_DFS(edges_f, seeds, acct, internal_accounts, budget_nodes=30000):
    hub = pick_hub(acct, internal_accounts)
    nodes, eds = bridging_C_with_dfs(
        edges_df=edges_f, seeds_df=seeds, hub_df=hub,
        component_depth=2, bridge_depth=3, max_neighbors_per_node=15, batch_size=200
    )
    if budget_nodes is not None and nodes.count() > budget_nodes:
        keep = nodes.limit(budget_nodes)
        eds = eds.join(keep.select(F.col("account").alias("src")), "src", "inner") \
                 .join(keep.select(F.col("account").alias("dst")), "dst", "inner")
        nodes = keep
    return nodes, eds


# ============================================================
# 4) Benchmark + PrettyTable
# ============================================================
def bench(name, fn):
    t0 = time.perf_counter()
    nodes_df, edges_df = fn()
    # materializa
    n_nodes = nodes_df.count()
    n_edges = edges_df.count()
    t1 = time.perf_counter()
    return {
        "name": name,
        "secs": (t1 - t0),
        "nodes": n_nodes,
        "edges": n_edges
    }

def pretty_print_benchmark(rows):
    try:
        from prettytable import PrettyTable
        pt = PrettyTable()
        pt.field_names = ["Heurística", "Búsqueda", "Segundos", "Nodos", "Aristas"]
        for r in rows:
            heur, search = r["name"].split("_", 1)
            pt.add_row([heur, search, f"{r['secs']:.2f}", f"{r['nodes']:,}", f"{r['edges']:,}"])
        print(pt)
    except Exception:
        # fallback simple si no está prettytable
        print("PrettyTable no disponible; imprimiendo formato simple.")
        print("Heurística | Búsqueda | Segundos | Nodos | Aristas")
        for r in rows:
            heur, search = r["name"].split("_", 1)
            print(f"{heur:9s} | {search:7s} | {r['secs']:.2f} | {r['nodes']:,} | {r['edges']:,}")


# ============================================================
# 5) Ejecutar todo (A/B/C x BFS/DFS)
# ============================================================
BUDGET = 30000

results = []

results.append(bench("A_BFS", lambda: run_A_BFS(edges_f, seeds, budget_nodes=BUDGET)))
results.append(bench("A_DFS", lambda: run_A_DFS(edges_f, seeds, budget_nodes=BUDGET)))

results.append(bench("B_BFS", lambda: run_B_BFS(edges_f, acct, internal_accounts, budget_nodes=BUDGET)))
results.append(bench("B_DFS", lambda: run_B_DFS(edges_f, acct, internal_accounts, budget_nodes=BUDGET)))

results.append(bench("C_BFS", lambda: run_C_BFS(edges_f, seeds, acct, internal_accounts, budget_nodes=BUDGET)))
results.append(bench("C_DFS", lambda: run_C_DFS(edges_f, seeds, acct, internal_accounts, budget_nodes=BUDGET)))

# Tabla comparativa solicitada (tiempo + nodos; agrego aristas)
pretty_print_benchmark(results)

# Si quieres además ordenar por nodos o por tiempo:
# for r in sorted(results, key=lambda x: x["secs"]): print(r)
